# Mini-projet : modèle de Leontief

| Module | Durée | Prérequis |
|---|---|---|
| M4 – Inverse, déterminant, rang | 150 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le conseil régional prépare son budget. La région compte 1,2 million d'habitants et 85 000 entreprises ; son économie est découpée en trois grands secteurs : **agriculture**, **industrie** et **services**. Le taux de chômage est de 7,4 %.

Chaque secteur a besoin des autres pour produire. Pour fabriquer 1 € de produits, l'industrie achète par exemple de l'énergie et des pièces (à l'industrie elle-même), des matières premières agricoles, des services de transport et de comptabilité. Ces besoins sont résumés dans un **tableau des coefficients techniques** (fichier `donnees/economie_region.csv`). On connaît aussi la **demande finale** prévue pour l'an prochain : ce que les ménages et les clients extérieurs achèteront.

**Les questions de l'élue chargée de l'économie :**

1. Quelle production totale chaque secteur doit-il réaliser pour satisfaire la demande finale ?
2. Si les exportations industrielles augmentent de 10 millions d'euros, de combien chaque secteur doit-il augmenter sa production ?
3. Quel secteur faut-il soutenir en priorité pour « entraîner » le reste de l'économie ?

**Activité** : Quelles informations de l'énoncé sont utiles au calcul ? Lesquelles ne le sont pas ? Quelles hypothèses ce modèle fait-il sur l'économie ?

Indiquez vos réponses et justifiez-les.

Ce modèle a valu le prix Nobel d'économie à Wassily Leontief, en 1973. Les valeurs utilisées ici sont **illustratives**.

## ② Estimer avant de calculer

La demande finale totale est de 370 millions d'euros. La production totale nécessaire est-elle plus petite, égale ou plus grande ? Pariez sur un montant.

In [ ]:
pari_production_totale = 400    # en millions d'euros (modifiez)

## ③ Abstraire

Notons $\mathbf{x}$ les productions des trois secteurs, $A$ la matrice des coefficients techniques et $\mathbf{d}$ la demande finale. Le coefficient $a_{ij}$ est ce que le secteur $j$ achète au secteur $i$ pour produire 1 €.

![Schéma : les trois secteurs se fournissent mutuellement, avec les coefficients techniques, et chacun alimente la demande finale](../../assets/schemas/schema-leontief.svg)

Chaque secteur doit produire de quoi fournir les autres secteurs **et** la demande finale :

$$
\mathbf{x} = A\,\mathbf{x} + \mathbf{d}
\quad\Longleftrightarrow\quad
(I - A)\,\mathbf{x} = \mathbf{d}
\quad\Longleftrightarrow\quad
\mathbf{x} = (I - A)^{-1}\,\mathbf{d}
$$

La matrice $L = (I - A)^{-1}$ s'appelle l'**inverse de Leontief**. Ici, on a besoin de $L$ elle-même, et pas seulement de la solution : la colonne $j$ de $L$ indique ce que chaque secteur doit produire en plus quand la demande finale du secteur $j$ augmente de 1 €.

**Une autre lecture : les vagues successives.** Pour livrer $\mathbf{d}$, il faut d'abord produire $\mathbf{d}$. Mais cela demande des fournitures $A\mathbf{d}$, qui demandent elles-mêmes $A^2\mathbf{d}$, et ainsi de suite :

$$
\mathbf{x} = \mathbf{d} + A\mathbf{d} + A^2\mathbf{d} + A^3\mathbf{d} + \cdots
\qquad\text{donc}\qquad
(I - A)^{-1} = I + A + A^2 + A^3 + \cdots
$$

Cette série converge quand les vagues s'amortissent. C'est le cas, par exemple, si chaque secteur dépense en fournitures moins d'1 € pour produire 1 € : toutes les sommes de colonnes de $A$ sont alors inférieures à 1.

**Activité** : Avec seulement deux secteurs, $A = \begin{pmatrix} 0{,}2 & 0{,}3 \\ 0{,}4 & 0{,}1 \end{pmatrix}$ et $\mathbf{d} = (10\,;\,20)$, calculez $(I - A)^{-1}$ avec la formule du notebook 4.1, puis $\mathbf{x}$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

donnees = pd.read_csv("donnees/economie_region.csv", index_col="fournisseur")
donnees

In [ ]:
secteurs = ["Agriculture", "Industrie", "Services"]
A = donnees[secteurs].to_numpy()
d = donnees["demande_finale"].to_numpy()
print("Sommes des colonnes de A :", {s: round(float(v), 2) for s, v in zip(secteurs, A.sum(axis=0))})

### Question 1 : la production nécessaire

In [ ]:
I = np.eye(3)
x = np.linalg.solve(I - A, d)
L = np.linalg.inv(I - A)

for s, xi, di in zip(secteurs, x, d):
    print(f"{s:12s} demande finale {di:6.1f} M€  ->  production {xi:6.1f} M€")
print(f"\nTotal : demande {d.sum():.0f} M€, production {x.sum():.1f} M€")
print("\nInverse de Leontief L :\n", pd.DataFrame(L.round(3), index=secteurs, columns=secteurs))

### Corroborer : la série des vagues

In [ ]:
vague = d.copy()
somme = np.zeros(3)
ecarts = []
for k in range(30):
    somme = somme + vague
    ecarts.append(np.linalg.norm(somme - x))
    vague = A @ vague

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.semilogy(range(1, 31), ecarts, "o-", ms=4)
ax.set_xlabel("nombre de vagues additionnées"); ax.set_ylabel("écart avec la solution (M€)")
ax.set_title("d + Ad + A²d + … converge vers (I − A)⁻¹ d")
plt.tight_layout(); plt.show()
print("Après 30 vagues :", somme.round(2), "   solution directe :", x.round(2))

### Questions 2 et 3 : les effets d'entraînement

In [ ]:
delta_d = np.array([0, 10, 0])            # + 10 M€ d'exportations industrielles
delta_x = L @ delta_d
print("Hausse de production (M€) :", {s: round(float(v), 2) for s, v in zip(secteurs, delta_x)}, f" total {delta_x.sum():.2f} M€")

multiplicateurs = L.sum(axis=0)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(secteurs, multiplicateurs, color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"]])
for i, m in enumerate(multiplicateurs):
    ax.text(i, m + 0.03, f"{m:.2f}", ha="center")
ax.set_ylabel("production totale induite par 1 € de demande")
ax.set_title("Multiplicateurs de production (somme des colonnes de L)")
ax.set_ylim(0, max(multiplicateurs) * 1.2)
plt.tight_layout(); plt.show()

### Les coefficients sont-ils fiables ?

Les coefficients techniques sont estimés à partir d'enquêtes : une incertitude de 10 % est réaliste. Faisons varier chaque coefficient au hasard de ±10 % et observons la production obtenue.

In [ ]:
rng = np.random.default_rng(7)
tirages = np.array([np.linalg.solve(I - A * rng.uniform(0.9, 1.1, size=A.shape), d) for _ in range(3000)])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.boxplot(tirages, widths=0.5)
ax.set_xticks([1, 2, 3], secteurs)
ax.scatter([1, 2, 3], x, color=LIOUAI["jaune"], zorder=3, label="coefficients du tableau")
ax.set_ylabel("production nécessaire (M€)"); ax.legend()
ax.set_title("Production quand chaque coefficient varie de ±10 %")
plt.tight_layout(); plt.show()
for s, col in zip(secteurs, tirages.T):
    print(f"{s:12s} : de {np.percentile(col, 5):6.1f} à {np.percentile(col, 95):6.1f} M€ (90 % des tirages)")

### Une économie qui ne tient pas

Imaginons une industrie très gourmande en ses propres produits : pour produire 1 €, elle consomme $a$ € d'industrie, avec $a$ de plus en plus grand.

In [ ]:
for a_ind in [0.30, 0.75, 0.85, 0.95]:
    A2 = A.copy(); A2[1, 1] = a_ind
    x2 = np.linalg.solve(I - A2, d)
    rayon = max(abs(np.linalg.eigvals(A2)))
    print(f"a = {a_ind:.2f} : somme de la colonne Industrie {A2[:, 1].sum():.2f}, "
          f"production {x2.round(0)}, rayon spectral {rayon:.3f}")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_production_totale} M€ ; production totale nécessaire : {x.sum():.0f} M€")

**Activité** :

1. Pourquoi la production totale dépasse-t-elle largement la demande finale ? Que représente l'écart ?
2. Interprétez la colonne Industrie de $L$ dans le langage de l'élue.
3. Quel secteur a le plus grand multiplicateur ? Est-ce forcément celui qu'il faut soutenir ? Quelles informations manquent pour décider (emplois créés, importations, pollution…) ?
4. Avec $a = 0{,}95$, le calcul donne des productions **négatives**. Que signifie ce résultat absurde ? Remarquez qu'avec $a = 0{,}75$ ou $0{,}85$, la colonne dépasse déjà 1 mais l'économie fonctionne encore : la condition « sommes de colonnes inférieures à 1 » suffit, mais elle n'est pas nécessaire. Le bon critère, le **rayon spectral**, sera étudié au module M7.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèses fragiles.** Le modèle suppose des coefficients **fixes** : pour produire deux fois plus, il faut exactement deux fois plus de chaque fourniture, sans économies d'échelle ni limite de capacité. Il ignore aussi les importations. Classez ces hypothèses de la plus à la moins gênante pour la question 2.
- **Vérifier autrement.** Vérifiez que $\mathbf{x} - A\mathbf{x} = \mathbf{d}$, et que la somme des vagues donne le même résultat que le calcul direct.
- **Communiquer.** Rédigez pour l'élue une note de cinq lignes au plus, sans formule : production nécessaire, effet de +10 M€ d'exportations industrielles, et fiabilité de ces chiffres.
- **Généraliser.** Écrivez une fonction `leontief(A, d)` qui renvoie la production nécessaire, et qui **refuse** de calculer si l'économie n'est pas productive ou si une production est négative. Testez-la sur une économie à 5 secteurs de votre invention.

In [ ]:
def leontief(A, d):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def leontief(A, d):
    A, d = np.asarray(A, dtype=float), np.asarray(d, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] != d.shape[0]:
        raise ValueError("A doit être carrée et de même taille que d.")
    if (A < 0).any() or (d < 0).any():
        raise ValueError("Les coefficients et la demande doivent être positifs.")
    if max(abs(np.linalg.eigvals(A))) >= 1:
        raise ValueError("Économie non productive : rayon spectral de A supérieur ou égal à 1.")
    x = np.linalg.solve(np.eye(len(d)) - A, d)
    assert np.allclose(x - A @ x, d)
    return x

print(leontief(A, d).round(1))
A5 = np.random.default_rng(1).uniform(0, 0.15, size=(5, 5))
print("Économie à 5 secteurs :", leontief(A5, [10, 20, 30, 40, 50]).round(1))
try:
    A_bad = A.copy(); A_bad[1, 1] = 0.95
    leontief(A_bad, d)
except ValueError as erreur:
    print("Refus :", erreur)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Filtrer l'information pertinente": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Formuler une hypothèse au-delà de l'étude": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Corroborer ses résultats": 1,
    "Expliquer par écrit": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M4-projet-leontief", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")